# Brawl Stars Bot — Сборка APK v2

Запускай ячейки по порядку. Каждый этап проверяй результат перед следующим.

In [ ]:
# Ячейка 1: Системные зависимости
import subprocess, time

print('[1/4] Установка системных пакетов...')
!apt-get update -qq
!apt-get install -y -qq git zip unzip openjdk-17-jdk-headless python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev libltdl-dev 2>&1 | grep -E 'Setting up|already|installed'

print('[2/4] Проверка Java...')
!java -version 2>&1

print('[3/4] Установка buildozer...')
!pip install -q buildozer cython 2>&1 | grep -v 'already satisfied'

print('[4/4] Проверка buildozer...')
!buildozer --version 2>&1

print('\n=== Готово! ===')

In [ ]:
# Ячейка 2: Загрузка проекта
from google.colab import files
import zipfile, os, glob

print('Загрузи BrawlStarsBot_android.zip:')
uploaded = files.upload()

for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall('.')
        print(f'Распаковано: {fn}')

# Ищем проект
spec_files = glob.glob('**/buildozer.spec', recursive=True)
if spec_files:
    project_dir = os.path.dirname(os.path.abspath(spec_files[0]))
    os.chdir(project_dir)
    print(f'\nПроект: {os.getcwd()}')
    print(f'Файлы: {os.listdir(".")}')
    print(f'images/: {os.listdir("images") if os.path.exists("images") else "НЕТ"}')
    print(f'\n=== Готово! ===')
else:
    print('ОШИБКА: buildozer.spec не найден! Загрузи правильный zip.')

In [ ]:
# Ячейка 3: Предустановка Android SDK (самый долгий этап, ~15-20 мин)
import os

print(f'Текущая папка: {os.getcwd()}')
print(f'buildozer.spec: {os.path.exists("buildozer.spec")}')
print()

# Buildozer сам скачает SDK при первом запуске.
# Запускаем с --verbose чтобы видеть прогресс.
# Используем nohup чтобы не отвалилось по таймауту.

print('=== Запуск buildozer (первый раз ~20-30 мин) ===')
print('Если Colab оборвётся —.sdk уже будет скачан, повторная сборка будет быстрой.')
print()

!buildozer -v android debug 2>&1 | tee /tmp/build_log.txt | grep -E 'Check|Download|Build|ERROR|WARN|SUCCESS|Compiling|Installing|Generating|Packaging|Run'

In [ ]:
# Ячейка 4: Скачивание APK
import os, glob

apk_files = glob.glob('**/*.apk', recursive=True)
bin_dir = os.path.join('bin', '')

if apk_files:
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / 1024 / 1024
        print(f'APK: {apk} ({size_mb:.1f} МБ)')
    from google.colab import files
    files.download(apk_files[-1])
else:
    print('APK не найден.')
    print('\nЛог сборки (последние 30 строк):\n')
    if os.path.exists('/tmp/build_log.txt'):
        with open('/tmp/build_log.txt') as f:
            lines = f.readlines()
            for line in lines[-30:]:
                print(line, end='')
    else:
        !ls -la bin/ 2>/dev/null || echo 'Папки bin/ нет'
        !ls -la .buildozer/ 2>/dev/null | head -20